# Population H3 Inspection Notebook

This notebook inspects the production population output produced by:

```bash
human build population
```

It does not fetch Census data or rebuild the pipeline. It reads the flat H3 res 7 population Parquet, recreates H3 geometries for inspection, optionally aggregates to a coarser H3 resolution, and writes HTML maps to `maps/population/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Literal

import folium
import geopandas as gpd
import h3
import matplotlib as mpl
import numpy as np
import pandas as pd
from branca.element import MacroElement, Template
from shapely.geometry import Polygon

## Configuration

Edit `MAP_VARIABLE`, `MAP_RESOLUTION`, and `MAP_AGGREGATION` to inspect a specific feature. Use `MAP_AGGREGATION = "auto"` for the notebook's default behavior:

- `sum` for population count columns
- `mean` for distance, density, boolean/share, and proximity-weight columns

In [ ]:
OUTPUT_FILENAMES = {
    "us": "us_population_h3_r7.parquet",
    "canada": "canada_population_h3_r7.parquet",
}
INCLUDE_CANADA_OUTPUT = True

SOURCE_RESOLUTION = 7
MAP_VARIABLE = "population"
MAP_RESOLUTION = 7
MAP_AGGREGATION: Literal["auto", "sum", "mean"] = "auto"
MAP_SOURCE_FILTER: str | None = None  # e.g. "canada", "us", or None for combined
MAP_COLOR_TRANSFORM: Literal["linear", "log1p"] = "linear"
DROP_ZERO_MAP_VALUES = False
CLEAR_OLD_MAPS = True

# Optional batch maps near the end of the notebook.
MAP_SPECS = [
    {"variable": "population", "resolution": 7, "aggregation": "sum"},
    {"variable": "population", "resolution": 7, "aggregation": "sum", "source": "canada", "color_transform": "log1p"},
    {"variable": "water_weighted_population_50mi", "resolution": 7, "aggregation": "sum", "source": "canada", "color_transform": "log1p"},
    {"variable": "water_weighted_population_25mi", "resolution": 7, "aggregation": "sum"},
    {"variable": "water_weighted_population_50mi", "resolution": 7, "aggregation": "sum"},
    {"variable": "distance_to_water_miles", "resolution": 7, "aggregation": "mean"},
    {"variable": "water_proximity_weight_50mi", "resolution": 7, "aggregation": "mean"},
    {"variable": "population", "resolution": 6, "aggregation": "sum"},
    {"variable": "water_weighted_population_50mi", "resolution": 6, "aggregation": "sum"},
    {"variable": "water_proximity_weight_50mi", "resolution": 6, "aggregation": "mean"},
]

In [ ]:
MAPS_DIR

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Use the configured toolkit workspace."""
    from human.core.config.paths import project_root
    return project_root()


def population_output_paths(project_root: Path) -> dict[str, Path]:
    processed_dir = project_root / "data" / "processed" / "domain" / "human" / "demography_and_presence" / "population"
    paths = {}
    us_path = processed_dir / OUTPUT_FILENAMES["us"]
    if not us_path.exists():
        message = (
            f"Could not find US population output: {us_path}"
            "\nRun `human build population` from the project root first."
        )
        raise FileNotFoundError(message)
    paths["us"] = us_path

    canada_path = processed_dir / OUTPUT_FILENAMES["canada"]
    if INCLUDE_CANADA_OUTPUT and canada_path.exists():
        paths["canada"] = canada_path
    elif INCLUDE_CANADA_OUTPUT:
        print(f"Canada output not found yet, skipping BC map layer: {canada_path}")
    return paths


def maps_dir_for_population(project_root: Path) -> Path:
    maps_dir = project_root / "notebooks" / "analysis" / "maps" / "population"
    maps_dir.mkdir(parents=True, exist_ok=True)
    return maps_dir


PROJECT_ROOT = find_project_root()
POPULATION_PATHS = population_output_paths(PROJECT_ROOT)
MAPS_DIR = maps_dir_for_population(PROJECT_ROOT)

PROJECT_ROOT, POPULATION_PATHS, MAPS_DIR

In [ ]:
if CLEAR_OLD_MAPS:
    for html_path in MAPS_DIR.glob("*.html"):
        html_path.unlink()
    print(f"Cleared old population HTML maps from {MAPS_DIR}")

## Load And Inspect The Flat Output

In [ ]:
def normalize_population_output(df: pd.DataFrame, source: str) -> pd.DataFrame:
    """Normalize US and Canada population outputs for shared inspection maps."""
    out = df.copy()
    out["source_output"] = source
    if "population_2020" in out.columns:
        out["population"] = out["population_2020"]
        out["population_round"] = out["population_2020_round"]
        out["population_density_per_km2"] = out["population_density_2020_per_km2"]
        out["region_label"] = out.get("state_abbr", "US")
    elif "population_2021" in out.columns:
        out["population"] = out["population_2021"]
        out["population_round"] = out["population_2021_round"]
        out["population_density_per_km2"] = out["population_density_2021_per_km2"]
        out["region_label"] = out.get("province_abbr", "CA")
    else:
        raise ValueError(f"No recognized population column in {source} output")
    return out


pieces = []
for source, path in POPULATION_PATHS.items():
    df = pd.read_parquet(path)
    print(f"Loaded {len(df):,} H3 rows from {source}: {path}")
    pieces.append(normalize_population_output(df, source))

population_df = pd.concat(pieces, ignore_index=True)
print(f"Combined rows before H3 map aggregation: {len(population_df):,}")
print(f"Unique H3 cells: {population_df['h3'].nunique():,}")
population_df.head()

In [ ]:
summary_cols = [
    "population",
    "population_density_per_km2",
    "distance_to_water_miles",
    "distance_to_water_capped_miles",
    "water_proximity_weight_25mi",
    "water_proximity_weight_50mi",
    "water_weighted_population_25mi",
    "water_weighted_population_50mi",
]
existing_summary_cols = [col for col in summary_cols if col in population_df.columns]
population_df[existing_summary_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
qa = {
    "rows": len(population_df),
    "unique_h3": population_df["h3"].nunique(),
    "h3_resolution_values": sorted(population_df["h3_resolution"].unique().tolist()),
    "total_population": float(population_df["population"].sum()),
    "populated_cells": int((population_df["population"] > 0).sum()),
}
for source, group in population_df.groupby("source_output"):
    qa[f"{source}_rows"] = int(len(group))
    qa[f"{source}_population"] = float(group["population"].sum())
for threshold in (25, 50, 75, 100):
    within_col = f"within_{threshold}mi_water"
    weighted_col = f"water_weighted_population_{threshold}mi"
    if within_col in population_df.columns:
        qa[f"population_within_{threshold}mi_water"] = float(
            population_df.loc[population_df[within_col], "population"].sum()
        )
    if weighted_col in population_df.columns:
        qa[f"weighted_population_{threshold}mi"] = float(population_df[weighted_col].sum())

pd.Series(qa)

## H3 Geometry And Aggregation Helpers

In [ ]:
def h3_cell_to_polygon(cell: str) -> Polygon:
    """Convert an H3 cell ID to a WGS84 Shapely polygon across h3-py versions."""
    if hasattr(h3, "cell_to_boundary"):
        boundary = h3.cell_to_boundary(cell)
        return Polygon([(lng, lat) for lat, lng in boundary])
    if hasattr(h3, "h3_to_geo_boundary"):
        return Polygon(h3.h3_to_geo_boundary(cell, geo_json=True))
    raise RuntimeError("Could not find a compatible H3 boundary function.")


def h3_parent(cell: str, resolution: int) -> str:
    """Return parent H3 cell across h3-py versions."""
    if hasattr(h3, "cell_to_parent"):
        return h3.cell_to_parent(cell, resolution)
    if hasattr(h3, "h3_to_parent"):
        return h3.h3_to_parent(cell, resolution)
    raise RuntimeError("Could not find a compatible H3 parent function.")


def default_aggregation(variable: str) -> Literal["sum", "mean"]:
    if variable in {"population", "population_round", "population_2020", "population_2021"}:
        return "sum"
    if variable.startswith("water_weighted_population_"):
        return "sum"
    return "mean"


def aggregate_for_map(
    df: pd.DataFrame,
    variable: str,
    resolution: int,
    aggregation: Literal["auto", "sum", "mean"] = "auto",
    source_resolution: int = SOURCE_RESOLUTION,
    source_filter: str | None = None,
) -> gpd.GeoDataFrame:
    """Prepare a geometry dataframe for one map variable and target H3 resolution."""
    if variable not in df.columns:
        raise ValueError(f"Variable not found: {variable}")
    if source_filter is not None:
        df = df[df["source_output"] == source_filter].copy()
        if df.empty:
            raise ValueError(f"No rows found for source_filter={source_filter!r}")
    if resolution > source_resolution:
        raise ValueError(f"Cannot aggregate from H3 res {source_resolution} to finer res {resolution}.")
    if aggregation == "auto":
        aggregation = default_aggregation(variable)
    if aggregation not in {"sum", "mean"}:
        raise ValueError("aggregation must be one of: auto, sum, mean")

    work = df[["h3", variable]].copy()
    work[variable] = pd.to_numeric(work[variable], errors="coerce")
    if resolution < source_resolution:
        work["map_h3"] = work["h3"].map(lambda cell: h3_parent(cell, resolution))
    else:
        work["map_h3"] = work["h3"]

    # Always group, even at source resolution, so overlapping US/Canada H3 cells aggregate.
    mapped = work.groupby("map_h3", as_index=False).agg(value=(variable, aggregation))
    mapped["geometry"] = mapped["map_h3"].map(h3_cell_to_polygon)
    gdf = gpd.GeoDataFrame(mapped, geometry="geometry", crs="EPSG:4326")
    gdf["variable"] = variable
    gdf["h3_resolution"] = resolution
    gdf["aggregation"] = aggregation
    gdf["source_filter"] = source_filter or "combined"
    return gdf

## Build One Inspection Map

This section uses the configurable `MAP_VARIABLE`, `MAP_RESOLUTION`, and `MAP_AGGREGATION` values above.

In [ ]:
map_gdf = aggregate_for_map(population_df, MAP_VARIABLE, MAP_RESOLUTION, MAP_AGGREGATION, source_filter=MAP_SOURCE_FILTER)
map_gdf[["map_h3", "value", "variable", "h3_resolution", "aggregation"]].head()

In [ ]:
map_gdf["value"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
def _format_legend_value(value: float) -> str:
    if abs(value) >= 1000:
        return f"{value:,.0f}"
    if abs(value) >= 10:
        return f"{value:,.1f}"
    return f"{value:,.3g}"


def _add_clean_legend(
    m: folium.Map,
    caption: str,
    colors: list[str],
    vmin: float,
    vmax: float,
    n_labels: int = 6,
) -> None:
    """Add a compact custom legend instead of Folium's crowded class legend."""
    gradient = ", ".join(f"{color} {i / (len(colors) - 1) * 100:.2f}%" for i, color in enumerate(colors))
    label_values = np.linspace(vmin, vmax, n_labels)
    labels_html = "".join(f"<span>{_format_legend_value(v)}</span>" for v in label_values)
    template = f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        top: 18px;
        right: 28px;
        z-index: 9999;
        width: 420px;
        background: rgba(255, 255, 255, 0.88);
        padding: 10px 12px 8px 12px;
        border: 1px solid rgba(0, 0, 0, 0.22);
        border-radius: 4px;
        box-shadow: 0 1px 4px rgba(0, 0, 0, 0.2);
        font-family: Arial, sans-serif;
        color: #222;
    ">
        <div style="font-weight: 600; font-size: 12px; margin-bottom: 6px;">{caption}</div>
        <div style="height: 14px; background: linear-gradient(to right, {gradient}); border: 1px solid rgba(0,0,0,0.25);"></div>
        <div style="display: flex; justify-content: space-between; font-size: 11px; margin-top: 4px;">{labels_html}</div>
    </div>
    {{% endmacro %}}
    """
    legend = MacroElement()
    legend._template = Template(template)
    m.get_root().add_child(legend)


def write_html_map(
    gdf: gpd.GeoDataFrame,
    variable: str,
    resolution: int,
    aggregation: str,
    maps_dir: Path = MAPS_DIR,
    drop_zero_values: bool = DROP_ZERO_MAP_VALUES,
    bins: int = 100,
    color_transform: Literal["linear", "log1p"] = MAP_COLOR_TRANSFORM,
) -> Path:
    """Write an interactive HTML map with 100 discrete RdBu_r bins and a clean legend."""
    maps_dir.mkdir(parents=True, exist_ok=True)
    safe_variable = variable.replace("/", "_").replace(" ", "_")
    source_label = str(gdf["source_filter"].iloc[0]) if "source_filter" in gdf.columns else "combined"
    suffix = "nonzero" if drop_zero_values else "all"
    out_path = maps_dir / f"{safe_variable}_H{resolution}_{aggregation}_{source_label}_{color_transform}_{suffix}.html"

    plot_gdf = gdf.to_crs("EPSG:4326").copy()
    plot_gdf["value"] = plot_gdf["value"].replace([np.inf, -np.inf], np.nan)
    plot_gdf = plot_gdf[plot_gdf["value"].notna()].copy()
    if drop_zero_values:
        plot_gdf = plot_gdf[plot_gdf["value"] != 0].copy()
    if plot_gdf.empty:
        raise ValueError(f"No mappable values for {variable}; try DROP_ZERO_MAP_VALUES = False")
    if color_transform == "linear":
        plot_gdf["color_value"] = plot_gdf["value"]
        inverse_color_value = lambda x: x
    elif color_transform == "log1p":
        if (plot_gdf["value"] < 0).any():
            raise ValueError("log1p color transform requires non-negative values")
        plot_gdf["color_value"] = np.log1p(plot_gdf["value"])
        inverse_color_value = np.expm1
    else:
        raise ValueError("color_transform must be one of: linear, log1p")

    vmin = float(plot_gdf["color_value"].min())
    vmax = float(plot_gdf["color_value"].max())
    cmap = mpl.colormaps["RdBu_r"].resampled(bins)
    colors = [mpl.colors.to_hex(cmap(i)) for i in range(bins)]
    if np.isclose(vmin, vmax):
        bin_edges = np.array([vmin, vmax + 1e-9])
    else:
        bin_edges = np.linspace(vmin, vmax, bins + 1)

    def color_for_value(value: float) -> str:
        if np.isclose(vmin, vmax):
            return colors[bins // 2]
        idx = int(np.searchsorted(bin_edges, value, side="right") - 1)
        idx = max(0, min(idx, bins - 1))
        return colors[idx]

    bounds = plot_gdf.total_bounds
    center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
    m = folium.Map(
        location=center,
        zoom_start=7,
        tiles="CartoDB positron",
        control_scale=True,
        prefer_canvas=True,
    )

    geojson = folium.GeoJson(
        plot_gdf,
        name=f"{variable} H{resolution} {aggregation}",
        style_function=lambda feature: {
            "fillColor": color_for_value(feature["properties"]["color_value"]),
            "color": "#333333",
            "weight": 0.15,
            "fillOpacity": 0.65,
            "opacity": 0.35,
        },
        highlight_function=lambda feature: {
            "weight": 1.0,
            "fillOpacity": 0.85,
            "opacity": 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["map_h3", "value", "variable", "aggregation", "source_filter"],
            aliases=["H3", "Value", "Variable", "Aggregation", "Source"],
            localize=True,
        ),
    )
    geojson.add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    _add_clean_legend(
        m,
        caption=f"{variable} ({aggregation}, H{resolution}, {source_label}, {color_transform}, {bins} bins)",
        colors=colors,
        vmin=float(inverse_color_value(vmin)),
        vmax=float(inverse_color_value(vmax)),
    )
    folium.LayerControl(collapsed=True).add_to(m)
    m.save(out_path)
    return out_path


single_map_path = write_html_map(
    map_gdf,
    MAP_VARIABLE,
    MAP_RESOLUTION,
    map_gdf["aggregation"].iloc[0],
    color_transform=MAP_COLOR_TRANSFORM,
)
single_map_path

## Batch QA Maps

These defaults focus on raw population, water-weighted population, distance to water, and proximity weights. Edit `MAP_SPECS` in the configuration cell to add/remove variables or change target H3 resolution.

In [ ]:
written_maps = []
for spec in MAP_SPECS:
    variable = spec["variable"]
    resolution = int(spec.get("resolution", SOURCE_RESOLUTION))
    aggregation = spec.get("aggregation", "auto")
    source_filter = spec.get("source")
    color_transform = spec.get("color_transform", MAP_COLOR_TRANSFORM)
    gdf = aggregate_for_map(population_df, variable, resolution, aggregation, source_filter=source_filter)
    actual_aggregation = gdf["aggregation"].iloc[0]
    out_path = write_html_map(gdf, variable, resolution, actual_aggregation, color_transform=color_transform)
    written_maps.append(
        {
            "variable": variable,
            "resolution": resolution,
            "aggregation": actual_aggregation,
            "source": source_filter or "combined",
            "color_transform": color_transform,
            "path": str(out_path),
        }
    )

pd.DataFrame(written_maps)

## Notes

- For H3 resolutions below 7, values are aggregated to parent cells.
- Population count fields use `sum`; distance and proximity fields use `mean` unless overridden.
- Parent-cell geometries are full H3 cells and are intended for inspection, not as a replacement for the production flat output.